# Combining GSE129363 and GSE278526: is ERG different in T2D endothelial cells?

**Author's note (junior bioinformatician):** every code cell has a short explanation above it, comments on the code lines
inside it, and, where the cell prints something, a note on what the output means.

## The question
Is the gene **ERG** (an endothelial transcription factor) expressed differently in endothelial cells of people with
type 2 diabetes (T2D) compared with controls, and does it differ between subcutaneous (SAT) and visceral (VAT) fat?

## The idea in plain words
Both datasets were already analysed in their own notebooks. Each gave us, for every gene, a *log2 fold change*
(how much higher or lower the gene is in T2D than in Control) and a *p-value*. Here we put the two answers side by side
and, where both are trustworthy, combine them into one answer.

**Why we do not just merge the cells:** the two studies used different sample preparation and sequencing depth, and in
GSE278526 each depot/group is one 10x library. Merging cells would mix study differences with disease differences.
So we combine the *results*, not the cells. This is called a **meta-analysis**.

## Words used in this notebook
| Term | Meaning |
|---|---|
| log2 fold change (logFC) | 0 = no change; +1 = twice as high in T2D; -1 = half as high in T2D |
| p-value | how surprising the difference would be if there were really no difference |
| FDR | p-value corrected for testing many genes at once (smaller = more convincing) |
| SE (standard error) | how precise an estimate is (smaller = more precise) |
| pseudobulk | counts added up per donor, so each donor counts once (not each cell) |
| SAT / VAT | subcutaneous / visceral (omental) fat |
| reads | the number of sequencing counts seen for a gene |

**Sign convention:** positive log2FC = higher in T2D. Negative = lower in T2D.

**How to use it:** run the cells from top to bottom. Only the paths in section 1 need editing.

## 1. Settings
Everything you might want to change is in this section, so you never need to edit the code further down.

**Cell annotation:** Load the plotting package. `ggplot2` is the only package needed; everything else is base R.

In [ ]:
library(ggplot2)   # used later for the forest plots

**Cell annotation:** Set the paths to the two result tables and the output folder. **Edit these three lines if your files are elsewhere.**

In [ ]:
# result table written by the GSE129363 notebook
file_129363 <- "results_v3/DE/Panel_DE_T2D_vs_Control_all_subsets.csv"
# result table written by the GSE278526 notebook
file_278526 <- "erg_panel_results_v2/panel_DE_T2D_vs_Control.csv"
# folder where this notebook saves its tables and figures
out_dir     <- "combined_results"

**Cell annotation:** Set the rules for a trustworthy result. A result must pass all three to be combined.

In [ ]:
min_reads  <- 50    # a gene needs at least this many reads in the tested samples (ERG is very low in 10x data)
min_donors <- 3     # at least this many donors in EACH group (Control and T2D); fewer is too unstable
fdr_cutoff <- 0.05  # significance threshold after correcting for testing many genes

*What you should see:* nothing is printed.

**Cell annotation:** List the eight panel genes and create the output folder.

In [ ]:
# ERG plus the seven other endothelial genes we were asked to check
panel <- c("ERG", "PECAM1", "CDH5", "NOS3", "VCAM1", "ICAM1", "HIF1A", "ANGPT2")
# make the output folder (does nothing if it already exists)
dir.create(out_dir, showWarnings = FALSE)

## 2. Read both result tables
Each notebook saved a table with one row per gene, tissue (SAT/VAT) and cell group. The column names are almost the same,
so we make them match and stack the two tables into one.

**Cell annotation:** Define a small function that reads one result table, fixes the one column name that differs, and checks that every needed column is there.

In [ ]:
read_results <- function(path, dataset_name) {
  d <- read.csv(path, stringsAsFactors = FALSE)           # read the csv file into a table
  names(d)[names(d) == "n_control"] <- "n_Control"        # one notebook used a lower-case name; make them match

  # the columns we need from each notebook
  needed <- c("depot", "subset", "gene", "n_Control", "n_T2D",
              "total_counts", "status", "logFC", "PValue")
  missing <- setdiff(needed, names(d))                    # which needed columns are absent?
  if (length(missing) > 0) stop(dataset_name, " is missing columns: ", paste(missing, collapse = ", "))

  d <- d[, needed]                                        # keep only the needed columns
  d$dataset <- dataset_name                               # remember which study each row came from
  d
}

*What you should see:* nothing is printed; the function is only defined here.

**Cell annotation:** Read the GSE129363 and GSE278526 tables.

In [ ]:
res_129363 <- read_results(file_129363, "GSE129363")      # table 1
res_278526 <- read_results(file_278526, "GSE278526")      # table 2

*What you should see:* nothing is printed. An error here means a path is wrong or a column is missing (the message names which).

**Cell annotation:** Stack the two tables, keep only the eight panel genes, and make sure the numbers are numeric.

In [ ]:
all_res <- rbind(res_129363, res_278526)                  # put table 2 under table 1
all_res <- all_res[all_res$gene %in% panel, ]             # keep only the eight genes of interest
all_res$logFC  <- as.numeric(all_res$logFC)               # make sure these are numbers, not text
all_res$PValue <- as.numeric(all_res$PValue)

**Cell annotation:** Check how many rows each dataset contributed and look at the first rows.

In [ ]:
print(table(all_res$dataset))   # number of rows per dataset
head(all_res)                   # first 6 rows

*What you should see:* a row count for each dataset, then six example rows. Rows with `NA` in logFC were not tested in that notebook.

## 3. Work out the standard error
To combine two estimates fairly we need to know how *precise* each one is (its standard error, SE). The notebooks only
saved the fold change and the p-value, so we recover the SE from them: `SE = |log2FC| / z`, where `z` is the z-score that
matches the two-sided p-value. This is an approximation. If you add an SE column to the two notebooks later, use that instead.

**Cell annotation:** Turn each p-value into a z-score (keeping p away from exactly 0 or 1 so the maths stays finite).

In [ ]:
p <- all_res$PValue                                  # copy the p-values
p <- pmin(pmax(p, 1e-300), 1 - 1e-12)                # keep p strictly between 0 and 1
z <- qnorm(p / 2, lower.tail = FALSE)                # z-score that matches a two-sided p-value

**Cell annotation:** Calculate the standard error from the fold change and z-score; unusable values become NA.

In [ ]:
all_res$se <- abs(all_res$logFC) / z                                 # SE = |logFC| / z
all_res$se[!is.finite(all_res$se) | all_res$se <= 0] <- NA          # mark impossible values as missing

**Cell annotation:** Look at the result for a few rows.

In [ ]:
head(all_res[, c("dataset", "gene", "depot", "subset", "logFC", "PValue", "se")])

*What you should see:* a small `se` means a precise estimate. Large `se` (for example above 1) means the estimate is very uncertain.

## 4. Decide which results are trustworthy
A result is only used if **all** of these are true: the test actually ran, both groups have enough donors, and the gene
has enough reads. ERG is detected at very low levels in 10x data, so a "significant" ERG result built on a handful of reads
is not reliable. Rows that fail are kept in the table with the reason, so nothing is hidden.

**Cell annotation:** Start by marking every row as eligible.

In [ ]:
all_res$reason <- "eligible"    # innocent until proven otherwise; failed rows get a reason below

**Cell annotation:** Find the rows that fail each rule.

In [ ]:
# rule 1: too few reads for this gene (or the read count is missing)
few_reads  <- which(is.na(all_res$total_counts) | all_res$total_counts < min_reads)
# rule 2: fewer than min_donors donors in the smaller of the two groups
few_donors <- which(pmin(all_res$n_Control, all_res$n_T2D) < min_donors)
# rule 3: no usable standard error
no_se      <- which(is.na(all_res$se))
# rule 4: the original notebook did not run the test
not_tested <- which(all_res$status != "tested" | is.na(all_res$logFC))

**Cell annotation:** Write the reason for each failed row. Later lines overwrite earlier ones, so the most basic problem is shown last.

In [ ]:
all_res$reason[few_reads]  <- paste0("fewer than ", min_reads, " reads")
all_res$reason[few_donors] <- paste0("fewer than ", min_donors, " donors in a group")
all_res$reason[no_se]      <- "no usable standard error"
all_res$reason[not_tested] <- "not tested"

**Cell annotation:** Count how many rows are usable in each dataset.

In [ ]:
print(table(all_res$dataset, all_res$reason))   # rows = dataset, columns = reason

*What you should see:* a table with one row per dataset. The `eligible` column is what goes into the combined result. A large number under 'fewer than 50 reads' means that dataset cannot say much about those genes.

## 5. Combine the two datasets
For each gene, tissue and cell group we look at the usable rows:

* **0 usable:** nothing to report.
* **1 usable:** we report that dataset's result alone. This is *not* a combined result, it has no replication.
* **2 usable:** inverse-variance meta-analysis (precise estimates count more), with a random-effects term so that
  disagreement between the datasets widens the uncertainty. `I2` is the % of variation due to disagreement
  (not reliable with only two datasets).

**Cell annotation:** Define the function that combines the usable rows of one gene/tissue/cell-group into a single estimate.

In [ ]:
pool_one <- function(d) {
  e <- d[d$reason == "eligible", ]                       # keep only the usable rows
  k <- nrow(e)                                           # how many datasets are usable (0, 1 or 2)
  out <- data.frame(k = k, datasets = paste(e$dataset, collapse = " + "),
                    logFC = NA, se = NA, p = NA, I2 = NA)    # empty answer to fill in
  if (k == 0) return(out)                                # nothing usable: return the empty answer

  if (k == 1) {                                          # only one dataset: just copy its result
    out$logFC <- e$logFC; out$se <- e$se; out$p <- e$PValue
    return(out)
  }

  w     <- 1 / e$se^2                                    # weight = precision (precise studies count more)
  fixed <- sum(w * e$logFC) / sum(w)                     # weighted average ("fixed effect")
  Q     <- sum(w * (e$logFC - fixed)^2)                  # how much the two datasets disagree
  tau2  <- max(0, (Q - (k - 1)) / (sum(w) - sum(w^2) / sum(w)))   # extra between-study variance (never negative)
  w2    <- 1 / (e$se^2 + tau2)                           # random-effects weights (disagreement widens the error)
  est   <- sum(w2 * e$logFC) / sum(w2)                   # combined log2 fold change
  se    <- 1 / sqrt(sum(w2))                             # its standard error

  out$logFC <- est
  out$se    <- se
  out$p     <- 2 * pnorm(-abs(est / se))                 # two-sided p-value of the combined estimate
  out$I2    <- max(0, (Q - (k - 1)) / Q) * 100           # % of variation due to disagreement
  out
}

*What you should see:* nothing is printed; the function is only defined here.

**Cell annotation:** Give every gene/tissue/cell-group combination a key and split the table into one piece per key.

In [ ]:
# one label per combination, e.g. "ERG|VAT|All_blood_ECs"
all_res$key <- paste(all_res$gene, all_res$depot, all_res$subset, sep = "|")
# one small table per label
pieces <- split(all_res, all_res$key)
length(pieces)   # how many combinations we will combine

*What you should see:* the number of gene / tissue / cell-group combinations.

**Cell annotation:** Run the combining function on every piece and stack the answers.

In [ ]:
pooled <- do.call(rbind, lapply(names(pieces), function(key) {   # loop over every combination
  r <- pool_one(pieces[[key]])                                   # combine its datasets
  r$key <- key                                                   # remember which combination it was
  r
}))

**Cell annotation:** Split the key back into separate gene, tissue and cell-group columns.

In [ ]:
parts <- strsplit(pooled$key, "|", fixed = TRUE)          # cut "ERG|VAT|All_blood_ECs" at each "|"
pooled$gene   <- sapply(parts, function(x) x[1])          # 1st piece = gene
pooled$depot  <- sapply(parts, function(x) x[2])          # 2nd piece = tissue
pooled$subset <- sapply(parts, function(x) x[3])          # 3rd piece = cell group

**Cell annotation:** Add the 95% confidence interval for each combined estimate.

In [ ]:
pooled$ci_low  <- pooled$logFC - 1.96 * pooled$se         # lower end of the 95% interval
pooled$ci_high <- pooled$logFC + 1.96 * pooled$se         # upper end of the 95% interval

## 6. Correct for testing many genes
Testing many genes will give some false positives by chance, so we adjust the p-values (Benjamini-Hochberg FDR).
Only results that really combine two datasets are adjusted and can be called significant.

**Cell annotation:** Adjust the p-values of the combined (two-dataset) results.

In [ ]:
pooled$FDR <- NA                                                  # start empty
two_sets <- which(pooled$k >= 2 & !is.na(pooled$p))               # only truly combined results
pooled$FDR[two_sets] <- p.adjust(pooled$p[two_sets], method = "BH")   # Benjamini-Hochberg correction

**Cell annotation:** Write a plain-language call for each row.

In [ ]:
pooled$call <- ifelse(pooled$k == 0, "not testable",                        # no usable dataset
               ifelse(pooled$k == 1, "single dataset (no replication)",     # one dataset only
               ifelse(!is.na(pooled$FDR) & pooled$FDR < fdr_cutoff,
                      "significant (FDR < 0.05)", "not significant")))      # two datasets: test the FDR

**Cell annotation:** Sort the table (panel genes first, in order) and count the calls.

In [ ]:
pooled <- pooled[order(match(pooled$gene, panel), pooled$depot, pooled$subset), ]   # tidy order
pooled$key <- NULL                                                                  # no longer needed
table(pooled$call)                                                                  # how many of each call

*What you should see:* a count for each kind of call. 'Single dataset' and 'not testable' are common when one dataset has low counts; they are honest, not failures.

## 7. Result for ERG
Read the `call` column first. If it says *single dataset*, ERG could only be judged in one study and the combined
answer is not available yet.

**Cell annotation:** Show the ERG rows of the combined table, rounded for reading.

In [ ]:
# keep only ERG and the columns worth reading
erg <- pooled[pooled$gene == "ERG",
              c("depot", "subset", "k", "datasets", "logFC", "ci_low", "ci_high", "p", "FDR", "call")]
erg[, c("logFC", "ci_low", "ci_high")] <- round(erg[, c("logFC", "ci_low", "ci_high")], 2)   # 2 decimals
erg$p   <- signif(erg$p, 3)                                                                 # 3 significant digits
erg$FDR <- signif(erg$FDR, 3)
print(erg, row.names = FALSE)

*What you should see:* one row per tissue and cell group. `logFC` is T2D relative to Control. If the confidence interval (`ci_low` to `ci_high`) includes 0, ERG is not clearly different.

**Cell annotation:** List the ERG rows that were left out and the reason, so nothing is hidden.

In [ ]:
left_out <- all_res[all_res$gene == "ERG" & all_res$reason != "eligible",
                    c("dataset", "depot", "subset", "total_counts", "reason")]
print(left_out, row.names = FALSE)

*What you should see:* which ERG results were excluded and why (usually too few reads). Report this next to the main result.

## 8. Forest plot for ERG
Each dot is one dataset's estimate with its 95% confidence interval. If the line crosses 0, the data are consistent with
no change. Faded dots were left out of the combined result (too few reads or donors). A black diamond is a combined
estimate (only drawn when both datasets were usable).

**Cell annotation:** Define a function that draws a forest plot for one gene.

In [ ]:
make_forest <- function(gene_name, subsets = NULL, title = "") {
  d <- all_res[all_res$gene == gene_name & !is.na(all_res$se), ]     # this gene, rows with an SE
  if (!is.null(subsets)) d <- d[d$subset %in% subsets, ]             # optionally limit the cell groups
  d$row  <- paste(d$depot, gsub("_", " ", d$subset))                 # label for the y axis, e.g. "VAT All blood ECs"
  d$low  <- d$logFC - 1.96 * d$se                                    # left end of the 95% interval
  d$high <- d$logFC + 1.96 * d$se                                    # right end of the 95% interval
  d$used <- ifelse(d$reason == "eligible", "used", "left out")       # used rows solid, others faded

  g <- ggplot(d, aes(x = logFC, y = row, colour = dataset, alpha = used)) +
    geom_vline(xintercept = 0, linetype = "dashed") +                # dashed line = no change
    geom_errorbar(aes(xmin = low, xmax = high), orientation = "y", width = 0.25,
                  position = position_dodge(0.6)) +                  # confidence interval lines
    geom_point(size = 3, position = position_dodge(0.6)) +           # the estimates (dots)
    scale_alpha_manual(values = c("used" = 1, "left out" = 0.35), name = NULL) +   # fade the excluded ones
    scale_colour_manual(values = c("GSE129363" = "#2A7F8E", "GSE278526" = "#C8622D"), name = NULL) +
    labs(x = paste(gene_name, "log2 fold change, T2D vs Control (95% CI)"), y = NULL, title = title) +
    theme_bw(base_size = 12) +
    theme(legend.position = "bottom")

  # add a black diamond where two datasets were combined
  pd <- pooled[pooled$gene == gene_name & pooled$k >= 2, ]
  if (!is.null(subsets)) pd <- pd[pd$subset %in% subsets, ]
  if (nrow(pd) > 0) {
    pd$row <- paste(pd$depot, gsub("_", " ", pd$subset))
    g <- g + geom_errorbar(data = pd, aes(xmin = ci_low, xmax = ci_high, y = row), orientation = "y",
                           width = 0.2, colour = "black", inherit.aes = FALSE) +
             geom_point(data = pd, aes(x = logFC, y = row), shape = 18, size = 5,
                        colour = "black", inherit.aes = FALSE)
  }
  g    # return the finished plot
}

*What you should see:* nothing is printed; the function is only defined here.

**Cell annotation:** Draw the ERG forest plot at a readable size.

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 6)       # size of the plot shown in the notebook
p_erg <- make_forest("ERG", title = "ERG: T2D vs Control, both datasets")
p_erg                                                      # display the plot

*What you should see:* one row per tissue and cell group. Dots right of the dashed line mean higher in T2D, left means lower. Lines that cross the dashed line mean no clear change. Wide lines mean an uncertain estimate.

## 9. The other panel genes (all blood endothelial cells)
Same idea for the rest of the gene panel, using the broadest endothelial group so that the read counts are highest.

**Cell annotation:** Prepare the data: keep the all-blood-EC rows and calculate the confidence interval for each dot.

In [ ]:
panel_rows <- all_res[all_res$subset == "All_blood_ECs" & !is.na(all_res$se), ]   # broadest EC group only
panel_rows$row  <- panel_rows$depot                                                # one row per tissue
panel_rows$low  <- panel_rows$logFC - 1.96 * panel_rows$se                         # interval, left end
panel_rows$high <- panel_rows$logFC + 1.96 * panel_rows$se                         # interval, right end
panel_rows$used <- ifelse(panel_rows$reason == "eligible", "used", "left out")     # solid vs faded
panel_rows$gene <- factor(panel_rows$gene, levels = panel)                         # keep the gene order

**Cell annotation:** Draw one small plot per gene.

In [ ]:
options(repr.plot.width = 11, repr.plot.height = 9)        # plot size in the notebook
p_panel <- ggplot(panel_rows, aes(x = logFC, y = row, colour = dataset, alpha = used)) +
  geom_vline(xintercept = 0, linetype = "dashed") +        # dashed line = no change
  geom_errorbar(aes(xmin = low, xmax = high), orientation = "y", width = 0.25, position = position_dodge(0.6)) +
  geom_point(size = 2.5, position = position_dodge(0.6)) +
  facet_wrap(~ gene, ncol = 2, scales = "free_x") +        # one small plot per gene, own x axis
  scale_alpha_manual(values = c("used" = 1, "left out" = 0.35), name = NULL) +
  scale_colour_manual(values = c("GSE129363" = "#2A7F8E", "GSE278526" = "#C8622D"), name = NULL) +
  labs(x = "log2 fold change, T2D vs Control (95% CI)", y = NULL,
       title = "Gene panel in all blood endothelial cells") +
  theme_bw(base_size = 11) + theme(legend.position = "bottom")
p_panel                                                    # display the plot

*What you should see:* eight small plots (one per gene) with SAT and VAT. Each x axis is scaled separately, so compare direction and whether the line crosses 0, not the size of the bars between genes.

## 10. Save the results

**Cell annotation:** Save the three result tables as CSV files.

In [ ]:
# the combined results for every gene
write.csv(pooled, file.path(out_dir, "combined_panel_results.csv"), row.names = FALSE)
# the same, ERG only
write.csv(pooled[pooled$gene == "ERG", ], file.path(out_dir, "combined_ERG_results.csv"), row.names = FALSE)
# every row from both datasets with the reason it was used or left out
write.csv(all_res[, names(all_res) != "key"], file.path(out_dir, "per_dataset_results_with_reasons.csv"), row.names = FALSE)

**Cell annotation:** Save the two figures as 300 dpi PNG files.

In [ ]:
ggsave(file.path(out_dir, "ERG_forest_combined.png"), p_erg, width = 10, height = 6, dpi = 300, bg = "white")
ggsave(file.path(out_dir, "panel_forest_all_blood_ECs.png"), p_panel, width = 11, height = 9, dpi = 300, bg = "white")

**Cell annotation:** Print where the files were saved and the software versions used (for reproducibility).

In [ ]:
cat("Saved to:", normalizePath(out_dir), "\n")   # full path of the output folder
sessionInfo()                                      # R and package versions

*What you should see:* the output folder path, then the R and package versions. Keep these with your results.

## 11. How to read the result, and its limits

* **A result is only "combined" when both datasets were usable.** Otherwise the `call` column says *single dataset*.
* **ERG has very few reads in GSE129363**, so it is usually left out there. That is a limit of the data, not evidence
  that ERG does not change.
* **The standard errors are approximate** (recovered from the p-values).
* **With two datasets, `I2` is unreliable.** Do not over-interpret it.
* **GSE278526:** each depot and group is one 10x library, so disease and library effects cannot be separated.
* **GSE129363:** the diabetes labels differ between the annotation file and GEO for three people (E07, 04, 09).
  Say which label set you used when you report the result.
* A third dataset with better ERG detection (for example the nuclei dataset GSE268904) would make this much stronger.

## Output files
| File | Contents |
|---|---|
| `combined_panel_results.csv` | combined estimate, confidence interval, FDR and call for every gene, tissue and cell group |
| `combined_ERG_results.csv` | the same for ERG only |
| `per_dataset_results_with_reasons.csv` | every input row with the reason it was used or left out |
| `ERG_forest_combined.png` | forest plot for ERG (300 dpi) |
| `panel_forest_all_blood_ECs.png` | forest plots for all eight genes (300 dpi) |